# Data — analyzer

**Block 3 of 3 in the Data stage, step 3 of 8.** The other two blocks *produce* data; this notebook
is where it is *understood*, and where a feature either earns a backtest or is dropped before
anybody spends a week on it.

**In plain words:** build features and test whether they carry signal, before you model anything.

**It produces** the charts and the information-coefficient table in `Data/Analyzer/`.

**It prevents** a book built on a feature that never predicted anything.

> **Runs after `Data/refinery.py`, and before any experiment** — the predictions an experiment's
> blueprint makes are supposed to come from here.

```
Data/curator.py    + Data/Curator/custom_calculations.py    ->  Curator/Time_Series/   m_* + c_*
Data/refinery.py   + Data/Refinery/custom_calculations.py   ->  Refinery/Time_Series/  + r_*
Data/analyzer.ipynb                                         ->  Analyzer/  charts + the IC table
```

`Universe/universe.ipynb` profiles the *catalogue* — what exists, what is missing, when each
security becomes usable. This notebook looks at the **content**: what the data says, and whether
the signal built on it carries anything.

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells.

## 0 · Setup

Read the refined panel, and name **the columns this notebook reads, once, in this cell** — the
eligibility column, the features it eats, the cross-sectional candidates to screen. Everything
below re-runs unchanged when they change.

| Column family | Built by | Scope |
| --- | --- | --- |
| `m_*` | the provider, via the Curator | raw market data |
| `c_*` | `Curator/custom_calculations.py` | **per security** — one security's own history |
| `r_*` | `Refinery/custom_calculations.py` | **cross-sectional**, or with a setting an experiment sweeps |
| `current_*` | joined from the security master | today's classification, **not point-in-time** |

In [ ]:
# EXAMPLE-ONLY CELL
import os
import pathlib
import sys

import numpy
import pandas

NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = (
    NOTEBOOK_DIRECTORY
    if (NOTEBOOK_DIRECTORY / "Experiments").is_dir()
    else NOTEBOOK_DIRECTORY.parent
)
os.chdir(REPOSITORY_ROOT)
sys.path.insert(0, str(REPOSITORY_ROOT / "Experiments"))

import securities_panel  # noqa: E402 - the path above has to exist first

# The columns this notebook reads, named once. Everything below re-runs unchanged when they change.
SIGNAL_COLUMN = "r_trend_50_200"
LIQUIDITY_COLUMN = "r_liquidity_rank"
# The next experiment's signal: the twelve-month return, the most recent month left out.
MOMENTUM_COLUMN = "r_momentum_12_1"
PRICE_COLUMN = "m_close_dividend_and_split_adjusted"
RETURN_COLUMN = "c_log_returns_dividend_and_split_adjusted"
# The strategy selects from the most traded names, so the pool it actually chooses from is the top
# of the liquidity rank rather than the whole universe.
ELIGIBLE_RANK = 0.95
HORIZONS_IN_DAYS = (21, 63, 252)
ANALYZER_DIRECTORY = pathlib.Path("Data/Analyzer")

matrices = securities_panel.load_matrices((
    SIGNAL_COLUMN,
    LIQUIDITY_COLUMN,
    MOMENTUM_COLUMN,
    PRICE_COLUMN,
    RETURN_COLUMN,
))
# The experiment's window ends here, and what comes after is held out until its book is frozen: the
# analyzer reads nothing later, so no measurement below can have seen it.
ANALYSIS_END = pandas.Timestamp("2026-06-01")
signal = matrices[SIGNAL_COLUMN].loc[:ANALYSIS_END]
liquidity = matrices[LIQUIDITY_COLUMN].loc[:ANALYSIS_END]
price = matrices[PRICE_COLUMN].loc[:ANALYSIS_END]
momentum = matrices[MOMENTUM_COLUMN].reindex(index=price.index, columns=price.columns)
print(f"panel: {price.shape[0]} dates x {price.shape[1]} securities")
print(f"from {price.index.min().date()} to {price.index.max().date()}")

## 1 · What each stage contributed

The refined file is the Curator file plus columns, same rows. Confirming that here is what lets
every section below read one folder and forget the Curator exists.

Then **coverage**, per column. A column at 60% coverage is not quietly averaged over the 60%: say
so, and decide whether the gap is a warm-up, a late listing or a broken input.

In [ ]:
# EXAMPLE-ONLY CELL
# Coverage per column, so nothing below is quietly averaged over the part that exists.
coverage = pandas.DataFrame({
    "populated": {name: int(matrix.notna().sum().sum()) for name, matrix in matrices.items()},
    "cells": {name: int(matrix.size) for name, matrix in matrices.items()},
})
coverage["share"] = (coverage["populated"] / coverage["cells"]).round(3)
print(coverage)

# The signal's gap is the 200-day warm-up plus late listings, not a broken input: it should fill in
# as the panel ages.
signal_by_year = signal.notna().sum(axis=1).groupby(signal.index.year).mean().round(0)
print("securities with a usable signal, by year:")
print(signal_by_year.tail(26))

## 2 · What diversification is actually available

A strategy that chooses between securities is a bet that they do not all move together. Whether
that is true is measurable, and it decides how much the strategy can possibly add: **if everything
is one trade, choosing between them is theatre.**

Buy-and-hold return, volatility and worst day per security; the correlation matrix; the mean
off-diagonal correlation and the extreme pairs.

In [ ]:
# EXAMPLE-ONLY CELL
# If everything is one trade, choosing between securities is theatre. This is the measurement.
# fill_method=None: a gap must stay a gap. Padding it turns the first price after a halt into
# one enormous return, which then dominates every statistic computed from the series.
daily_returns = price.pct_change(fill_method=None)
survivors = daily_returns.columns[daily_returns.notna().sum() > 500]
sample = daily_returns[survivors]
correlations = sample.corr()
off_diagonal = ~numpy.eye(len(correlations), dtype=bool)
# Pairs that never traded on the same day have no correlation at all, and averaging over them
# as if they were zeros would understate how much the universe moves together.
mean_correlation = numpy.nanmean(correlations.to_numpy()[off_diagonal])
print(f"securities with more than 500 daily observations: {len(survivors)}")
print(f"mean pairwise correlation: {mean_correlation:.3f}")

per_security = pandas.DataFrame({
    "annualised_return": (sample.mean() * 252).round(3),
    "annualised_volatility": (sample.std() * (252 ** 0.5)).round(3),
    "worst_day": sample.min().round(3),
})
print(per_security.describe().round(3))

## 3 · Are the cross-sectional columns what they claim to be?

The Refinery asserts one property that would be silent if broken: **every rank is a percentile
taken inside a single date.** A rank computed over the pooled sample would drift as the universe's
composition changed, and nothing would raise an error.

Check it as an identity, not as a rule of thumb. A per-date percentile over *n* untied values has
mean exactly `(n + 1) / (2n)` — 0.542 on twelve securities, 0.5006 on eight hundred. Testing
against 0.5 looks like a small failure on every date of a narrow universe and passes on a wide one,
which is the worst possible failure mode.

In [ ]:
# EXAMPLE-ONLY CELL
# A per-date percentile over n untied values averages to (n + 1) / (2n), not to 0.5. Checking
# against 0.5 fails on every date of a narrow universe and passes on a wide one.
observed_mean = liquidity.mean(axis=1)
count_per_date = liquidity.notna().sum(axis=1)
expected_mean = (count_per_date + 1) / (2 * count_per_date)
difference = (observed_mean - expected_mean).abs()
tested = difference[count_per_date > 0]
print(f"dates tested: {len(tested)}")
print(f"largest deviation from the identity: {tested.max():.6f}")
print(f"dates off by more than 0.001: {int((tested > 0.001).sum())}")

# The identity assumes no ties, so where it misses, the question is whether ties explain it. A
# security that has not traded for a whole quarter has a traded value of exactly zero, and ties
# with any other in the same state; tied securities share a rank.
worst_date = tested.idxmax()
ranked = liquidity.loc[worst_date].dropna()
tied = len(ranked) - ranked.nunique()
print(f"worst date {worst_date.date()}: {len(ranked)} ranked, {tied} sharing a rank")

## 4 · Information coefficient — which features predict returns

**The section that decides things.** For each feature and horizon, the information coefficient is
the cross-sectional rank correlation between the feature on date *t* and the forward return from
*t* to *t+h*, computed **per date** and then averaged. Per date is what keeps it causal: the
correlation only ever compares securities observable at the same moment.

- **IC** — the mean daily correlation. The sign matters as much as the size: a negative IC means
  the feature works *inverted*.
- **IR** = IC / IC standard deviation — the consistency of the edge, which is what survives into a
  portfolio.
- **The share of dates with the expected sign**, beside the IR, and the overlap stated. A forward
  window of *h* days, measured every day, shares *h − 1* days with the next, so the dates are not
  independent observations and an IR counted over them overstates the evidence.

Compute it over the whole panel and, separately, over the **eligible pool** the strategy actually
selects from. The second is the one that matters: a feature can behave differently inside an
already-filtered group.

> **The IC table is a screening tool, not evidence.** An information ratio scales with the square
> root of the number of independent bets, so on a narrow universe read these as directional. **A
> feature that fails here does not get a book built on it**; one that passes has earned a backtest,
> not a belief.

In [ ]:
# EXAMPLE-ONLY CELL
# The rank correlation between the feature on date t and the forward return from t to t+h, taken
# per date and then averaged. Per date is what keeps it causal.
eligible = liquidity >= ELIGIBLE_RANK
rows = []

for horizon in HORIZONS_IN_DAYS:
    forward_return = price.shift(-horizon) / price - 1

    for pool_name, mask in (("whole panel", None), ("eligible pool", eligible)):
        feature = signal if mask is None else signal.where(mask)
        target = forward_return if mask is None else forward_return.where(mask)
        daily_correlation = feature.corrwith(
            target,
            axis=1,
            method="spearman",
        ).dropna()
        information_coefficient = daily_correlation.mean()
        information_ratio = information_coefficient / daily_correlation.std()
        # The rule expects a positive coefficient. Consecutive windows share horizon - 1 days, so
        # the dates are not independent, and the share that had the expected sign sits beside the
        # ratio as the reading that does not pretend they are.
        expected_sign_share = (daily_correlation > 0).mean()
        rows.append({
            "feature": SIGNAL_COLUMN,
            "pool": pool_name,
            "horizon_days": horizon,
            "overlap_days": horizon - 1,
            "information_coefficient": round(information_coefficient, 4),
            "information_ratio": round(information_ratio, 3),
            "share_expected_sign": round(expected_sign_share, 3),
            "dates": len(daily_correlation),
        })

information_table = pandas.DataFrame(rows)
ANALYZER_DIRECTORY.mkdir(parents=True, exist_ok=True)
information_table.to_csv(ANALYZER_DIRECTORY / "information_coefficient.csv", index=False)
information_table

## 5 · The two questions any signal owes an answer to

Sections 0 to 4 are what any strategy needs. What belongs beside them depends on your signal, and
two are worth writing whatever it is.

**Does the signal separate anything?** Split forward return *and* forward volatility by the
signal's state. A signal can be worth trading on the second alone; it would not be the first.

**If the signal is fitted, what is look-ahead worth?** Read the same model causally and smoothed
and report the gap. It is the cheapest audit in the process and routinely the largest number in it:
the two series agree on most days and differ exactly at the turning points, which is where the
money is.

In [ ]:
# EXAMPLE-ONLY CELL
# Does the signal separate anything? Forward return AND forward volatility, split by the state the
# rule actually reads: the sign of the trend distance, inside the pool the rule selects from.
horizon = 21
forward_return = price.shift(-horizon) / price - 1
daily = price.pct_change(fill_method=None)
forward_volatility = daily.rolling(horizon).std().shift(-horizon) * (252 ** 0.5)
in_pool = liquidity >= ELIGIBLE_RANK
uptrend = (signal > 0) & in_pool
downtrend = (signal <= 0) & in_pool

separation = pandas.DataFrame({
    "forward_return_21d": {
        "50-day above 200-day": forward_return.where(uptrend).stack().mean(),
        "50-day below 200-day": forward_return.where(downtrend).stack().mean(),
    },
    "forward_volatility_21d": {
        "50-day above 200-day": forward_volatility.where(uptrend).stack().mean(),
        "50-day below 200-day": forward_volatility.where(downtrend).stack().mean(),
    },
    "observations": {
        "50-day above 200-day": int(uptrend.sum().sum()),
        "50-day below 200-day": int(downtrend.sum().sum()),
    },
})
print(separation.round(4))

# This signal is arithmetic on past prices, not a fitted model, so there is no smoothed reading to
# compare a causal one against: the look-ahead audit costs nothing here and is worth zero.
print("signal is unfitted: causal and smoothed readings are the same series")

In [ ]:
# EXAMPLE-ONLY CELL
# The rule reads the cross as a state, 1 above and 0 below, not as a distance. So the state's own
# coefficient sits beside the distance's, in the pool the rule selects from -- the owner's binary
# test -- and the question the first book raised is measured here: what a name earns after its
# cross breaks, against the rest of the pool on the same dates. A name is sold the day after a
# break at the earliest, so this is the most a rule that holds on can lose, not a cost it pays.
state = (signal > 0).astype(float).where(signal.notna())
state_rows = []

for state_horizon in HORIZONS_IN_DAYS:
    state_forward = price.shift(-state_horizon) / price - 1
    state_feature = state.where(eligible)
    state_correlation = state_feature.corrwith(
        state_forward.where(eligible),
        axis=1,
        method="spearman",
    ).dropna()
    state_rows.append({
        "feature": "cross state, 1 above and 0 below",
        "pool": "eligible pool",
        "horizon_days": state_horizon,
        "information_coefficient": round(state_correlation.mean(), 4),
        "information_ratio": round(state_correlation.mean() / state_correlation.std(), 3),
        "share_expected_sign": round((state_correlation > 0).mean(), 3),
        "dates": len(state_correlation),
    })

state_table = pandas.DataFrame(state_rows)
state_table.to_csv(ANALYZER_DIRECTORY / "cross_state.csv", index=False)
print(state_table.to_string(index=False))

# A break: the state was 1 yesterday and is 0 today, for a name in the pool.
broken = (state.shift(1) == 1) & (state == 0) & eligible
break_rows = []

for days_after in (5, 21, 63):
    after_break = price.shift(-days_after) / price - 1
    pool_average = after_break.where(eligible).mean(axis=1)
    excess = after_break.sub(pool_average, axis=0).where(broken).stack()
    break_rows.append({
        "days_after_break": days_after,
        "breaks": len(excess),
        "mean_excess_over_pool": round(excess.mean(), 4),
        "median_excess_over_pool": round(excess.median(), 4),
        "share_below_pool": round((excess < 0).mean(), 3),
    })

break_table = pandas.DataFrame(break_rows)
break_table.to_csv(ANALYZER_DIRECTORY / "cross_breaks.csv", index=False)
print(break_table.to_string(index=False))

In [ ]:
# EXAMPLE-ONLY CELL
# Twelve-month momentum, measured before Experiment 3's blueprint in a neighbour of the pool
# that experiment selects from -- the hundred most traded securities on each date, members or
# not -- and in the whole panel, over the two windows the experiments have used. Consecutive
# 21-day windows share 20 days, so the coefficient is also read on the first trading day of
# each month alone, where consecutive windows share only the days by which a month falls short
# of 21 trading days.
MOMENTUM_POOL_SIZE = 100
MOMENTUM_BOOK_SIZE = 20
MOMENTUM_WINDOWS = (
    ("2002 to 2016", pandas.Timestamp("2002-07-30"), pandas.Timestamp("2016-12-30")),
    ("2017 to 2026", pandas.Timestamp("2017-01-03"), pandas.Timestamp("2026-06-01")),
)
liquidity_order = liquidity.rank(axis=1, ascending=False, method="first")
momentum_pool = (liquidity_order <= MOMENTUM_POOL_SIZE) & momentum.notna()
month_numbers = pandas.Series(price.index.month, index=price.index)
month_starts = price.index[month_numbers.ne(month_numbers.shift(1))]
momentum_rows = []

for momentum_horizon in HORIZONS_IN_DAYS:
    momentum_forward = price.shift(-momentum_horizon) / price - 1

    for momentum_pool_name, momentum_mask in (
        ("whole panel", None),
        ("hundred most traded", momentum_pool),
    ):
        momentum_feature = momentum if momentum_mask is None else momentum.where(momentum_mask)
        momentum_target = (
            momentum_forward
            if momentum_mask is None
            else momentum_forward.where(momentum_mask)
        )
        momentum_daily = momentum_feature.corrwith(
            momentum_target,
            axis=1,
            method="spearman",
        ).dropna()

        for window_name, window_start, window_end in MOMENTUM_WINDOWS:
            in_window = momentum_daily.loc[window_start:window_end]
            at_month_starts = in_window.reindex(month_starts).dropna()
            momentum_rows.append({
                "feature": MOMENTUM_COLUMN,
                "pool": momentum_pool_name,
                "window": window_name,
                "horizon_days": momentum_horizon,
                "information_coefficient": round(in_window.mean(), 4),
                "share_expected_sign": round((in_window > 0).mean(), 3),
                "dates": len(in_window),
                "month_start_coefficient": round(at_month_starts.mean(), 4),
                "month_start_share_expected_sign": round((at_month_starts > 0).mean(), 3),
                "month_starts": len(at_month_starts),
            })

momentum_table = pandas.DataFrame(momentum_rows)
momentum_table.to_csv(ANALYZER_DIRECTORY / "momentum.csv", index=False)
print(momentum_table.to_string(index=False))

# The selection itself, without a book: on each month start, the twenty names of the pool with the
# highest momentum against the pool's own average over the next 21 days. No costs, no sizing, no
# lag: a measurement of the signal, never a return a book earned.
next_month = price.shift(-21) / price - 1
momentum_order = momentum.where(momentum_pool).rank(axis=1, ascending=False, method="first")
chosen = momentum_order <= MOMENTUM_BOOK_SIZE
selection_rows = []

for window_name, window_start, window_end in MOMENTUM_WINDOWS:
    starts = month_starts[(month_starts >= window_start) & (month_starts <= window_end)]
    pool_mean = next_month.where(momentum_pool).loc[starts].mean(axis=1)
    chosen_mean = next_month.where(chosen).loc[starts].mean(axis=1)
    spread = (chosen_mean - pool_mean).dropna()
    selection_rows.append({
        "window": window_name,
        "month_starts": len(spread),
        "mean_excess_over_pool": round(spread.mean(), 4),
        "median_excess_over_pool": round(spread.median(), 4),
        "share_above_pool": round((spread > 0).mean(), 3),
    })

selection_table = pandas.DataFrame(selection_rows)
selection_table.to_csv(ANALYZER_DIRECTORY / "momentum_selection.csv", index=False)
print(selection_table.to_string(index=False))

## 6 · Handoff

| Output | Consumed by |
| --- | --- |
| `Data/Refinery/Time_Series/` | **`Experiments/` — read this one** |
| `Data/Analyzer/` — the IC table | feature selection in every experiment |
| `Data/Analyzer/Charts/` | `FINDINGS_N.md` |

## What to write in the blueprint before running an experiment

**Write down what you expect before you run this.** A prediction made from the data and then
confirmed by the engine is the strongest methodological result an experiment can report; a number
found first and explained afterwards is a story.

Put the predictions this notebook licenses in `BLUEPRINT_N.md` **before** the backtest, each with
the section it came from and the observation that would falsify it. **Findings from this stage go
straight into `RESULTS.md`**, under *Before any experiment* — notebook outputs are stripped before
committing, so a measurement living only in a cell output does not survive the commit.

In [ ]:
# EXAMPLE-ONLY CELL
print(f"{ANALYZER_DIRECTORY / 'information_coefficient.csv'} -> BLUEPRINT_1.md predictions")
print("Findings from this stage go straight into RESULTS.md, under 'Before any experiment':")
print(information_table.to_string(index=False))
print(separation.round(4).to_string())

## 7 · Verify

**Assertions that raise when this stage's output is wrong**, read back from what this notebook
wrote to `Data/Analyzer/` rather than from the variables that wrote it. A notebook that reaches its
last cell is then one whose table can be quoted. At the least:

- the coefficient table has a row for every feature, pool and horizon, each with a finite
  coefficient, ratio and share of dates with the expected sign;
- no coefficient is averaged over no dates, nor over more dates than the panel has a forward
  return for at its horizon;
- the pool the strategy selects from never counts more dates than the whole panel at the same
  horizon, because it is a part of it.

In [ ]:
# EXAMPLE-ONLY CELL
# Read back the table this notebook wrote, and raise on the first thing that is wrong.
table_path = ANALYZER_DIRECTORY / "information_coefficient.csv"
written_table = pandas.read_csv(table_path)
measured = written_table[[
    "information_coefficient",
    "information_ratio",
    "share_expected_sign",
]]
dates_by_pool = written_table.pivot(
    index="horizon_days",
    columns="pool",
    values="dates",
)
# A forward return over h days needs a price h days later, so the last h dates have none.
dates_with_forward_return = pandas.Series(
    {
        horizon: len(price.index) - horizon
        for horizon in HORIZONS_IN_DAYS
    },
)
verifications = {
    "a row for every pool and horizon": len(written_table) == 2 * len(HORIZONS_IN_DAYS),
    "every coefficient, ratio and sign share is finite": bool(
        numpy.isfinite(measured).all(axis=None)
    ),
    "every sign share lies between 0 and 1": bool(
        measured["share_expected_sign"].between(0, 1).all()
    ),
    "every row averages over at least one date": bool((written_table["dates"] > 0).all()),
    "no row counts more dates than have a forward return": bool(
        (dates_by_pool.max(axis=1) <= dates_with_forward_return.loc[dates_by_pool.index]).all()
    ),
    "the eligible pool never counts more dates than the whole panel": bool(
        (dates_by_pool["eligible pool"] <= dates_by_pool["whole panel"]).all()
    ),
}
state_written = pandas.read_csv(ANALYZER_DIRECTORY / "cross_state.csv")
breaks_written = pandas.read_csv(ANALYZER_DIRECTORY / "cross_breaks.csv")
verifications["a cross-state row for every horizon, each finite"] = bool(
    len(state_written) == len(HORIZONS_IN_DAYS)
    and numpy.isfinite(state_written["information_coefficient"]).all()
)
verifications["every break row counts at least one break"] = bool(
    (breaks_written["breaks"] > 0).all()
)
momentum_written = pandas.read_csv(ANALYZER_DIRECTORY / "momentum.csv")
selection_written = pandas.read_csv(ANALYZER_DIRECTORY / "momentum_selection.csv")
verifications["a momentum row for every pool, window and horizon, each finite"] = bool(
    len(momentum_written) == 2 * len(MOMENTUM_WINDOWS) * len(HORIZONS_IN_DAYS)
    and numpy.isfinite(momentum_written["information_coefficient"]).all()
)
verifications["a selection row for every window, each over at least one month"] = bool(
    len(selection_written) == len(MOMENTUM_WINDOWS)
    and (selection_written["month_starts"] > 0).all()
)
failed = [
    name
    for name, passed in verifications.items()
    if not passed
]

if len(failed) > 0:
    message = f"the analyzer failed verification: {'; '.join(failed)}"

    raise AssertionError(message)

print(f"verified: {len(verifications)} checks on {table_path}")